In [1]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, Tool  
from agents.mcp import MCPServerStdio
from openai import AsyncOpenAI
from datetime import datetime, date
import asyncio

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

google_api_key = os.getenv('GOOGLE_API_KEY')
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3-flash-preview", openai_client=gemini_client)

openai_model = "gpt-5-mini"
project_path = os.path.abspath(os.path.join(os.getcwd()))

files_params = {
    "command": "npx",
    "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        project_path
    ]
}

file_server = MCPServerStdio(params=files_params,client_session_timeout_seconds=60)

instructions = """
You are a certified financial analyst and expert of SEC EDGAR filings.
You are given a financial statement XBRL concept and best match against a predifined income statement line 
item for a company. The file with the mapping is income_statement_xbrl_mapping.py. 
The file already has the mapping for the most common concepts.
Use MCP Server tools to access income_statement_xbrl_mapping.py"""

async def verify_concept_mapping(instructions) -> Agent:
    instructions = instructions
    verify_concept_mapping = Agent(
        name="Verify Concept Mapping Agent",
        instructions=instructions,
        mcp_servers=[file_server],
        model=openai_model
    )
    return verify_concept_mapping
    
concept_agent = await verify_concept_mapping(instructions)

concept = "SellingAndMarketingExpense"
user_instructions = f"""
You are given the XBR concept {concept}.
Map the given XBRL concept to the best match income statement category line item in the predifined mapping file
test_predifined_xbrl_mapping.py.
Return the income statement line item from the file_server
test_predifined_xbrl_mapping.py. Return on the line item without extra verbage.
If the concept is already mapped, just return "Already mapped" without any extra verbage.
"""

#START MCP SERVER
await file_server.connect()

#RUN AGENT
with trace("Verify Concept Mapping Agent"):
    result = await Runner.run(concept_agent, user_instructions)


In [2]:
print(result.final_output)

Already mapped


In [13]:
import os
from edgar import Company , set_identity
from dotenv import load_dotenv

# Load environment variables
load_dotenv()


# Set SEC identity to avoid 403 blocks
sec_identity = os.getenv("SEC_ID")

set_identity(sec_identity)

company = Company("MSFT")

filing = company.latest("10-K")



In [16]:
type(filing)

edgar.entity.filings.EntityFiling

In [39]:
from edgar import Company
from edgar.xbrl import XBRLS

company = Company("AAPL")
filings = company.get_filings(form="10-K" , year=2024)
xbrl = filing.xbrl()

facts = xbrl.facts

print(facts)


Facts for ╭───────────────────────────────────────────────── XBRL Document ─────────────────────────────────────────────────╮
│ MICROSOFT CORPORATION (MSFT) • CIK 0000789019                                                                   │
│                                                                                                                 │
│          Form:  10-K                                                                                            │
│ Fiscal Period:  Fiscal Year 2025 (ended Jun 30, 2025)                                                           │
│          Data:  1,829 facts • 441 contexts                                                                      │
│                                                                                                                 │
│ Periods Available for Statements:                                                                               │
│   Annual: FY 2025                                           

In [43]:
results = (xbrl.query()
            .by_concept("us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax")
           )

In [44]:
print(results)

╭────────────────────────────────────────────────── Facts Query ──────────────────────────────────────────────────╮
│ Use to_dataframe(columns) to get a DataFrame of the results.                                                    │
│                                                                                                                 │
│ e.g. query.to_dataframe('concept', 'value', 'period_end')                                                       │
│                                                                                                                 │
│ Available columns: 'concept, label, balance, preferred_sign, weight, value, numeric_value, period_key,          │
│ period_start, period_end, is_dimensioned, decimals, statement_type, statement_name, unit_ref, fiscal_period,    │
│ fiscal_year'                                                                                                    │
│                                                                       

In [30]:
print(statements)

Available Statements:

Financial Statements:
  3. Role StatementBALANCESHEETS
  5. Role StatementCASHFLOWSSTATEMENTS
  2. Role StatementCOMPREHENSIVEINCOMESTATEMENTS
  1. Role StatementINCOMESTATEMENTS
  92. Role DisclosureStockholdersEquityAdditionalInformationDetail
  6. Role StatementSTOCKHOLDERSEQUITYSTATEMENTS

Disclosures:
  41. DisclosureAccumulatedOtherComprehensiveIncomeLossTables
  70. DisclosureComponentsOfLongtermDebtDetail
  71. DisclosureComponentsOfLongtermDebtParentheticalDetail (Parenthetical)
  8. DisclosureCybersecurityRiskManagementStrategyAndGovernance
  57. DisclosureGainsLossesNetOfTaxOnDerivativeInstrumentsRecognizedInConsolidatedComprehensiveIncomeStatementsDetail
  56. DisclosureGainsLossesOnDerivativeInstrumentsRecognizedInOtherIncomeExpenseNetDetail
  103. DisclosureSegmentRevenueCostOfRevenueOperatingExpensesAndOperatingIncomeDetail
  63. DisclosureSupplementalConsolidatedFinancialResultsOnUnauditedProFormaBasisDetail
  84. DisclosureUnearnedRevenueRemainin

In [28]:
print(cashflow_stmt)

                                                                                                                                                      
                                                         MICROSOFT CORPORATION   MSFT                                                                 
                                                         CONSOLIDATED STATEMENT OF CASH FLOWS                                                         
                                                         Jun 30, 2023 to Jun 30, 2025                                                                 
                                                                                                                                                      
                                                                                                        Jun 30, 2025   Jun 30, 2024   Jun 30, 2023    
   ───────────────────────────────────────────────────────────────────────────────────────────

In [25]:
df = income_stmt.to_df()
df.head()






AttributeError: 'StitchedStatement' object has no attribute 'to_df'

In [2]:
# Now try extraction again
cashflow_df = await extract_cash_flow(
    filing,
    TICKER,
    FILING_TYPE,
    YEAR,
    QUARTER,
    use_ai_fallback=USE_AI_FALLBACK
)

NameError: name 'filing' is not defined

In [ ]:
print(result.final_output)
